In [33]:
import pandas as pd
from git import Repo
import requests
import base64
import os

In [34]:
with open('module1_basics/fibonacci_2.c', 'r') as file:
    contents = file.read()
    line_count = contents.count('\n')
    print("Total lines:", line_count)

Total lines: 70


In [35]:
contents = contents.split('\n')

In [36]:
logical_lines = line_count
contents_modified = []
for i in range(len(contents)):
    contents[i]=contents[i].replace(" ","")
    if len(contents[i])==0:
        #print(f"'{contents[i]}'  -- len=0")
        logical_lines-=1
    if len(contents[i])>0 and contents[i][0]=='/':
        #print(f"'{contents[i]}'  -- comment")
        logical_lines-=1
    #l= any(c.isalpha() for c in contents[i])
    #n= any(c.isdigit() for c in contents[i])
    
    #if (n or l):
    #    nb_lines+=1
logical_lines


29

In [37]:
with open('module1_basics/fibonacci_2.c', 'r') as file:
    contents = file.read()
    line_count = contents.count('\n')
    print("Total lines:", line_count)

Total lines: 70


In [38]:
contents = contents.split('\n')

In [39]:
mccabe = 1
for i in range(len(contents)):
    if 'if ' in contents[i] or 'for ' in contents[i] or 'while ' in contents[i] or 'case ' in contents[i] or '&&' in contents[i] or '||' in contents[i]:
        mccabe+=1
mccabe

7

# ----------------------------------------------------------------------------------------

In [40]:
def get_git_branches(repo_url):
    api_url = f"{repo_url.rstrip('/')}/info/refs?service=git-upload-pack"
    response = requests.get(api_url, stream=True)
    print(api_url)
    print(response)

    branches = []
    for line in response.iter_lines(decode_unicode=True):
        if line.startswith("refs/heads/"):
            branch_name = line.split("refs/heads/")[-1]
            branches.append(branch_name)
    
    return branches

In [41]:
repo_url = "https://github.com/miroslawstaron/software_metrics_course"
#repo_url = "https://github.com/torvalds/linux"
clone_dir_path = "/Users/lamartp/Documents/test"

In [42]:
clone_dir_path = clone_dir_path+"_SE_metrics"

In [43]:
#Repo.clone_from(repo_url, clone_dir_path)

In [44]:
repo = Repo(clone_dir_path)
#commit = repo.commit('master')

In [45]:
#create a list of all files in the repository
file_paths = []
for root, dirs, files in os.walk(repo.working_tree_dir):
    for file in files:
        if file.endswith('.c') or file.endswith('.java') or file.endswith('.py'):
            file_paths.append(os.path.relpath(os.path.join(root, file), repo.working_tree_dir))
            #print(os.path.relpath(os.path.join(root, file), repo.working_tree_dir))

In [46]:
file_metrics = {}

total_phys_lines = 0
total_logical_lines = 0

for file in file_paths:
    file_to_read = os.path.join(clone_dir_path, file)
    with open(file_to_read) as f:
        contents = f.read()

        phys_lines = contents.count('\n')

        logical_lines = phys_lines
        contents = contents.split('\n')
        mccabe = 1

        for i in range(len(contents)):
            if 'if ' in contents[i] or 'for ' in contents[i] or 'while ' in contents[i] or 'case ' in contents[i] or '&&' in contents[i] or '||' in contents[i]:
                mccabe+=1

            contents[i]=contents[i].replace(" ","")
            if len(contents[i])==0:
                logical_lines-=1
            if file.endswith('.c') and len(contents[i])>0 and contents[i][0]=='/':
                logical_lines-=1
            if file.endswith('.java') and len(contents[i])>0 and contents[i][0]=='/':
                logical_lines-=1
            if file.endswith('.py') and len(contents[i])>0 and contents[i][0]=='#':
                logical_lines-=1
        
        dict = {}
        dict["physical_lines", "logical_lines", "mccabe"] = phys_lines, logical_lines, mccabe
        file_metrics[file] = dict

        total_phys_lines += phys_lines
        total_logical_lines += logical_lines
    

In [47]:
file_metrics

{'module1_basics\\fibonacci_1.c': {('physical_lines',
   'logical_lines',
   'mccabe'): (35, 26, 4)},
 'module1_basics\\fibonacci_2.c': {('physical_lines',
   'logical_lines',
   'mccabe'): (70, 29, 7)},
 'module1_basics\\fibonacci_3.c': {('physical_lines',
   'logical_lines',
   'mccabe'): (54, 24, 10)},
 'module1_basics\\fibonacci_4.c': {('physical_lines',
   'logical_lines',
   'mccabe'): (79, 47, 11)},
 'module1_basics\\fibonacci_4.java': {('physical_lines',
   'logical_lines',
   'mccabe'): (82, 49, 11)},
 'module1_basics\\fibonacci_4.py': {('physical_lines',
   'logical_lines',
   'mccabe'): (72, 40, 12)},
 'module1_basics\\fibonacci_5.py': {('physical_lines',
   'logical_lines',
   'mccabe'): (39, 31, 6)},
 'module2_measurement_systems\\module_2_demos\\main.c': {('physical_lines',
   'logical_lines',
   'mccabe'): (5, 4, 1)}}